# Aggregating the fate accuracy prediction results
In this notebook, we will aggregate the fate accuracy prediction results across all the models.

For each model, we set up two training configuration: one on the PCA space and the other one is on the Diffusion Map space. Then for each config, we ran at least 3 repeats with different random seeds to account for the randomness in the training process.

In [2]:
%load_ext autoreload
%autoreload 2

import os, sys
import numpy as np
import pandas as pd
import json

In [3]:
os.chdir("/rds/user/wz369/hpc-work/PINN_dynamics")
higher_dir = '/rds/user/wz369/hpc-work'

# pseudodynamics+ (pdp+)

## PC 30

In [4]:
import re
def fate_eval_file_regex(filename):
    patterns = re.match(r"t(\S{,3})_n(\S{,3})_(\S{,3})_fate_eval.csv", filename).groups()
    return patterns

In [5]:
eval_dir = f"{higher_dir}/pseudodynamics_plus/results/pseudodynamics+/" 

PCconfig_repeats = [
    "klein_PC_30_lD1_lv1_lgNone",
    "klein_PC30_lD1_cfm1_lgNone",
    "klein_PC30_lD1_cfm1_lv1_lgNone",
    "klein_PC30_lD1_cfm2_lgNone",
    "klein_PC30_lD1_cfm10_lgNone",
    "klein_PC30_lD1_cfm10_lgNone_b1024"
    ]

# loops through all the eval files
results = []
for repeat in PCconfig_repeats:
    eval_subfolder = os.path.join(eval_dir, repeat)
    for file in os.listdir(eval_subfolder):
        if file.endswith("_fate_eval.csv"):
            int_time, noise, sim_fn = fate_eval_file_regex(file)
            # print(int_time, noise, sim_fn)
            df = pd.read_csv(os.path.join(eval_subfolder, file))
            df['sim_fn'] = sim_fn
            results.append(df)
            
# drop the intemediate simulation results
pdp_eval_results = pd.concat(results)
ranked_perform = pdp_eval_results.sort_values(
    by=['accuracy'], ascending=False
        ).drop_duplicates(subset=['model_dir'], keep='first')

In [6]:
ranked_perform.to_csv("scripts/pdp_ranked_perform_PC.csv")

## Diffusion map (10 dimension)

In [7]:
eval_dir = f"{higher_dir}/pseudodynamics_plus/results/pseudodynamics+/" 

DMconfig_repeats = [
    # "klein_DM_10_lD1_cfm1_lv1_lgNone_b512",
    # "klein_DM_10_lD1_cfm10_lgNone_b512",
    # "klein_DM_10_lD1_cfm10_lgNone_b1024",
    # "klein_DM_10_lD1_lv1_lgNone",
    # "klein_DM_10_lD1_lv10_lgNone",
    # "klein_DM_10_lD10_lv1_lgNone"
    "klein_DMscaled_10_cfm5_b512",
    "klein_DMscaled_10_cfm5_b1024",
    "klein_DMscaled_10_cfm10_b512",
    "klein_DMscaled_10_cfm10_b1024",
    ]

# loops through all the eval files
results = []
for repeat in DMconfig_repeats:
    eval_subfolder = os.path.join(eval_dir, repeat)
    for file in os.listdir(eval_subfolder):
        if file.endswith("_fate_eval.csv"):
            int_time, noise, sim_fn = fate_eval_file_regex(file)
            # print(int_time, noise, sim_fn)
            df = pd.read_csv(os.path.join(eval_subfolder, file))
            df['sim_fn'] = sim_fn
            results.append(df)
            
# drop the intemediate simulation results
pdp_eval_results = pd.concat(results)
DM_ranked_perform = pdp_eval_results.sort_values(
    by=['accuracy'], ascending=False
        ).drop_duplicates(subset=['model_dir'], keep='first')

DM_ranked_perform

,accuracy,pearson_r,pearson_p,n_start_cells,n_sims,k_nn,obsm_key,n_dims,t_end_norm,noise_scale,model_dir,sim_fn
0,0.479074,0.911810,0.000238,2031,100,15,DM_EigenVectors_scaled,10,0.5,0.5,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,sb
0,0.423929,0.750766,0.012338,2031,100,15,DM_EigenVectors_scaled,10,0.5,0.5,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,sb
0,0.422944,0.861627,0.001353,2031,100,15,DM_EigenVectors_scaled,10,0.5,0.5,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,sb
0,0.408173,0.844051,0.002134,2031,100,15,DM_EigenVectors_scaled,10,0.5,0.5,/rds/user/wz369/hpc-work/pseudodynamics_plus/l...,sb


In [8]:
DM_ranked_perform.iloc[0]['model_dir']

'/rds/user/wz369/hpc-work/pseudodynamics_plus/logs/klein_DMscaled_10_cfm5_b1024/pde_params_tsense/V0_config.json'

In [9]:
DM_ranked_perform.iloc[-2:]['model_dir'].values

array(['/rds/user/wz369/hpc-work/pseudodynamics_plus/logs/klein_DMscaled_10_cfm10_b1024/pde_params_tsense/V0_config.json',
       '/rds/user/wz369/hpc-work/pseudodynamics_plus/logs/klein_DMscaled_10_cfm5_b512/pde_params_tsense/V0_config.json'],
      dtype=object)

In [10]:
DM_ranked_perform.to_csv("scripts/pdp_ranked_perform_DM.csv")

# DeepRUOT

In [44]:
eval_dir = f"{higher_dir}/DeepRUOTv2/results/"
print(eval_dir)

/rds/user/wz369/hpc-work/DeepRUOTv2/results/


## PC 30

In [34]:
PC_repeats = [
    'klein_pca30', 'klein_pca30_s0', 'klein_pca30_s10', 'klein_pca30_s42'
]

DeepRUOTresults = []
for repeat in PC_repeats:
    print(repeat)
    pf = pd.read_csv(os.path.join(eval_dir, repeat, "eval_combined.csv"))
    
    DeepRUOTresults.append(pf)

DeepRUOT_pc = pd.concat(DeepRUOTresults) 
DeepRUOT_pc

klein_pca30
klein_pca30_s0
klein_pca30_s10
klein_pca30_s42


,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k
0,ode,0.360906,0.912658,11.860204,11.860204,2031,1,15
0,ode,0.334810,0.975341,11.517741,11.517741,2031,1,15
0,ode,0.341704,0.822840,11.682735,11.682735,2031,1,15
0,ode,0.324471,0.646240,11.315379,11.315379,2031,1,15


In [35]:
PC_repeats = [
    'klein_dm10', 'klein_dm10_s0', 'klein_dm10_s10', 'klein_dm10_s42'
]

DeepRUOTresults = []
for repeat in PC_repeats:
    print(repeat)
    pf = pd.read_csv(os.path.join(eval_dir, repeat, "eval_combined.csv"))
    
    DeepRUOTresults.append(pf)

DeepRUOT_dm = pd.concat(DeepRUOTresults) 
DeepRUOT_dm

klein_dm10
klein_dm10_s0
klein_dm10_s10
klein_dm10_s42


,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k
0,ode,0.076809,0.809221,0.006889,0.006889,2031,1,15
0,ode,0.164451,0.731430,0.007299,0.007299,2031,1,15
0,ode,0.084687,0.815819,0.007517,0.007517,2031,1,15
0,ode,0.147710,0.739281,0.006890,0.006890,2031,1,15


# scDiffeq

In [40]:
results = []
for seed in [0,1,2]:
    eval_dir = f"{higher_dir}/scDiffEq/results/seeds/seed_{seed}/pca30/plain_sde/fate_prediction_metrics/last"
    df = pd.read_csv(os.path.join(eval_dir, "eval_combined.csv"))
    results.append(df)

scDiffeq_pc = pd.concat(results)
scDiffeq_pc

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k
0,sde,0.385032,0.846869,10.490688,10.490688,2031,200,15
0,sde,0.377646,0.846252,10.662208,10.662208,2031,200,15
0,sde,0.373215,0.840821,10.662832,10.662832,2031,200,15


In [41]:
results = []
for seed in [0,1,2]:
    eval_dir = f"{higher_dir}/scDiffEq/results/seeds/seed_{seed}/dm10/plain_sde/fate_prediction_metrics/last"
    df = pd.read_csv(os.path.join(eval_dir, "eval_combined.csv"))
    results.append(df)

scDiffeq_dm = pd.concat(results)
scDiffeq_dm

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k
0,sde,0.434761,0.734353,0.007385,0.007385,2031,200,15
0,sde,0.274742,0.908217,0.007252,0.007252,2031,200,15
0,sde,0.383555,0.777750,0.007415,0.007415,2031,200,15


# TIGON
To run evaluation :

```shell
bash /rds/user/wz369/hpc-work/PINN_dynamics/scripts/TIGON/03_evaluate.sh
```

In [42]:
results = []
for repeat in ["model", "model_seed2", "model_seed3"]:
    df = pd.read_csv(f"{higher_dir}/PINN_dynamics/logs/TIGON/pca/{repeat}/eval_combined.csv")
    results.append(df)
TIGON_pc = pd.concat(results)
TIGON_pc

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k
0,ode,0.038897,0.980651,4.251537,12.15261,2031,1,15
0,ode,0.038897,0.982976,4.250278,12.14623,2031,1,15
0,ode,0.038897,0.982976,4.250278,12.14623,2031,1,15


In [43]:
results = []
for repeat in ["model", "model_seed2", "model_seed3"]:
    df = pd.read_csv(f"{higher_dir}/PINN_dynamics/logs/TIGON/dm/{repeat}/eval_combined.csv")
    results.append(df)
TIGON_dm = pd.concat(results)
TIGON_dm

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k
0,ode,0.078779,0.873658,2.837012,0.007061,2031,1,15
0,ode,0.036435,0.963030,2.803302,0.006978,2031,1,15
0,ode,0.036435,0.963030,2.803302,0.006978,2031,1,15


# MIOFlow

In [46]:
results = []
for repeat in [1,2,3]:
    df = pd.read_csv(f"logs/MIOFlow/klein_pca_gaga_latent15_run{repeat}/eval_combined.csv")
    results.append(df)
mioflow_pc = pd.concat(results)
mioflow_pc

,sim_mode,accuracy,pearson_r,w2_raw,n_start_cells,n_sims_fate,k
0,ode,0.288528,0.627970,55.566775,2031,1,15
0,ode,0.186608,0.088185,41.333348,2031,1,15
0,ode,0.265387,0.686167,45.062405,2031,1,15


# TrajectoryNet

In [48]:
results = []
for repeat in ['model','model_run2','model_run3','model_run4']:
    df = pd.read_csv(f"logs/TrajectoryNet/pca30/{repeat}/eval_combined.csv")
    results.append(df)

TJN_pc = pd.concat(results)
TJN_pc

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k
0,ode,0.092565,0.780078,4.696540,12.383142,2031,1,15
0,ode,0.084195,0.823549,4.762419,12.415095,2031,1,15
0,ode,0.117676,0.764594,4.759443,12.445251,2031,1,15
0,ode,0.092073,0.778220,4.810399,12.521586,2031,1,15


In [ ]:
# not yet evaluated
"logs/TrajectoryNet/dm10/model_run2/eval_combined.csv"

# PRESICENT

In [50]:
results = [] 
for seed in [0,2,42]:
    df = pd.read_csv(f"results/PRESCIENT/pca_run/growth_weights-softplus_1_500-1e-06/seed_{seed}/eval_combined.csv")
    results.append(df)

PRESCIENT_pc = pd.concat(results)
PRESCIENT_pc

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k
0,sde,0.523880,0.897716,17.746177,17.746177,2031,100,15
0,sde,0.520433,0.899075,17.633522,17.633522,2031,100,15
0,sde,0.523387,0.900645,17.728345,17.728345,2031,100,15


In [ ]:
results = [] 
for seed in [0,2,42]:
    df = pd.read_csv(f"results/PRESCIENT/dm_run/growth_weights-softplus_4_64-1e-06/seed_{seed}/eval_combined.csv")
    results.append(df)

PRESCIENT_dm = pd.concat(results)
PRESCIENT_dm

# SF2M 

In [54]:
results = [] 
for r in range(1,4):
    results.append(pd.read_csv(f"logs/sf2m/pca30/model_r{r}/eval_combined.csv"))

sf2m_pc = pd.concat(results)
sf2m_pc

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k
0,ode,0.330379,0.773490,4.014820,3.880834,2031,1,15
1,sde,0.339734,0.771635,3.974200,3.841227,2031,100,15
0,ode,0.330379,0.773490,4.014820,3.880834,2031,1,15
1,sde,0.340719,0.772129,3.974310,3.841405,2031,100,15
0,ode,0.330379,0.773490,4.014820,3.880834,2031,1,15
1,sde,0.338749,0.772471,3.975051,3.842134,2031,100,15


In [55]:
results = [] 
for r in range(1,4):
    results.append(pd.read_csv(f"logs/sf2m/dm10/model_r{r}/eval_combined.csv"))
sf2m_dm = pd.concat(results)
sf2m_dm

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k
0,ode,0.147218,0.024976,0.036775,0.000091,2031,1,15
1,sde,0.321024,0.728547,0.030503,0.000073,2031,100,15
0,ode,0.147218,0.024976,0.036775,0.000091,2031,1,15
1,sde,0.317578,0.733647,0.030550,0.000073,2031,100,15
0,ode,0.147218,0.024976,0.036775,0.000091,2031,1,15
1,sde,0.316593,0.731776,0.030636,0.000073,2031,100,15


# OTCFM

In [56]:
results = [] 
for r in range(1,4):
    results.append(pd.read_csv(f"logs/otcfm/pca30/model_r{r}/eval_combined.csv"))

sf2m_pc = pd.concat(results)
sf2m_pc

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k
0,ode,0.300345,0.772158,4.138142,3.972581,2031,1,15
0,ode,0.300345,0.772158,4.138142,3.972581,2031,1,15
0,ode,0.300345,0.772158,4.138142,3.972581,2031,1,15


In [57]:
results = [] 
for r in range(1,4):
    results.append(pd.read_csv(f"logs/otcfm/dm10/model_r{r}/eval_combined.csv"))
sf2m_dm = pd.concat(results)
sf2m_dm

,sim_mode,accuracy,pearson_r,w2_scaled,w2_raw,n_start_cells,n_sims_fate,k
0,ode,0.372723,0.848451,2.521864,0.006332,2031,1,15
0,ode,0.372723,0.848451,2.521864,0.006332,2031,1,15
0,ode,0.372723,0.848451,2.521864,0.006332,2031,1,15
